In [1]:
eval_session_path = "/home/ga53voq/master_thesis/logs/curriculum_final-2025-09-09_12-41-01/C5_eval_session_20250912_224208"

In [2]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from pathlib import Path

In [3]:
# Load the evaluation session data
session_summary_path = os.path.join(eval_session_path, "session_summary.json")

print(f"Loading data from: {eval_session_path}")

# Load session summary
with open(session_summary_path, 'r') as f:
    session_data = json.load(f)

print(f"Evaluation timestamp: {session_data['eval_timestamp']}")
print(f"Episodes per cave: {session_data['episodes_per_cave']}")
print(f"Number of caves: {len(session_data['caves'])}")
print(f"Caves directory: {session_data['caves_directory']}")
print(f"Max steps per episode: {session_data['max_steps']}")

Loading data from: /home/ga53voq/master_thesis/logs/curriculum_final-2025-09-09_12-41-01/C5_eval_session_20250912_224208
Evaluation timestamp: 20250912_224208
Episodes per cave: 3
Number of caves: 30
Caves directory: new_caves/goal_0.1
Max steps per episode: 7500


In [4]:
# Extract all reward data
all_rewards = []
cave_rewards = {}

for cave_data in session_data['caves']:
    cave_id = cave_data['cave_id']
    cave_rewards[cave_id] = []
    
    for episode in cave_data['episodes']:
        reward = episode['final_cumulative_reward']
        all_rewards.append(reward)
        cave_rewards[cave_id].append(reward)

print(f"Total episodes analyzed: {len(all_rewards)}")
print(f"Caves analyzed: {sorted(cave_rewards.keys())}")
print(f"First few rewards: {all_rewards[:10]}")
print(f"Reward range: [{min(all_rewards):.2f}, {max(all_rewards):.2f}]")

Total episodes analyzed: 90
Caves analyzed: [271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 297, 298, 299, 300]
First few rewards: [39.92646752594521, 37.25345664643646, 28.550333298917394, 23.62174335724103, 21.96086340556127, 46.38691010312823, 31.273345491378677, 21.73338200546641, 0.6981449077611614, 30.769127188963466]
Reward range: [-15.61, 91.97]


In [5]:
# Calculate per-cave statistics
cave_stats = {}
cave_means = []
cave_stds = []

print("\nPER-CAVE STATISTICS (Average over all episodes per cave)")
print("="*80)
print(f"{'Cave ID':<8} {'Mean Reward':<12} {'Std Dev':<12} {'Episodes':<10} {'Min':<10} {'Max':<10}")
print("-"*80)

for cave_id in sorted(cave_rewards.keys()):
    rewards = cave_rewards[cave_id]
    cave_mean = np.mean(rewards)
    cave_std = np.std(rewards)
    cave_min = np.min(rewards)
    cave_max = np.max(rewards)
    
    cave_stats[cave_id] = {
        'mean': cave_mean,
        'std': cave_std,
        'min': cave_min,
        'max': cave_max,
        'count': len(rewards)
    }
    
    cave_means.append(cave_mean)
    cave_stds.append(cave_std)
    
    print(f"{cave_id:<8} {cave_mean:<12.4f} {cave_std:<12.4f} {len(rewards):<10} {cave_min:<10.4f} {cave_max:<10.4f}")

print("-"*80)


PER-CAVE STATISTICS (Average over all episodes per cave)
Cave ID  Mean Reward  Std Dev      Episodes   Min        Max       
--------------------------------------------------------------------------------
271      35.2434      4.8569       3          28.5503    39.9265   
272      30.6565      11.1437      3          21.9609    46.3869   
273      17.9016      12.7730      3          0.6981     31.2733   
274      31.0854      10.2004      3          18.7537    43.7334   
275      34.6914      14.7865      3          13.8280    46.3468   
276      17.4217      11.6708      3          4.2633     32.6295   
277      28.2825      19.5284      3          2.4405     49.6404   
278      13.6706      19.3167      3          -10.7273   36.5120   
279      40.0245      20.9069      3          22.2280    69.3704   
280      31.6174      12.7338      3          17.9627    48.6127   
281      18.1507      10.4775      3          3.3466     26.0958   
282      50.9183      12.8268      3         

In [6]:
# Analyze maximum x-position and energy usage in each episode
print("="*60)
print("MAXIMUM X-POSITION AND ENERGY USAGE ANALYSIS")
print("="*60)

all_max_x_positions = []
cave_max_x_positions = {}
all_energy_usage = []
cave_energy_usage = {}

for cave_data in session_data['caves']:
    cave_id = cave_data['cave_id']
    cave_max_x_positions[cave_id] = []
    cave_energy_usage[cave_id] = []
    
    print(f"Processing Cave {cave_id}...")
    
    for episode in cave_data['episodes']:
        episode_file = os.path.join(eval_session_path, cave_data['dir'], episode['log_file'])
        
        # Load episode data
        with open(episode_file, 'r') as f:
            episode_data = json.load(f)
        
        # Find maximum x-position and last energy usage during the episode
        max_x_pos = 0.0  # Initialize to 0
        last_energy_usage = 0.0  # Initialize to 0
        
        for step_data in episode_data['steps']:
            if 'info' in step_data:
                # Track max x-position
                if 'max_x_position' in step_data['info']:
                    x_pos = step_data['info']['max_x_position']
                    if x_pos > max_x_pos:
                        max_x_pos = x_pos
                
                # Track energy usage (keep last value)
                if 'energy_usage' in step_data['info']:
                    last_energy_usage = step_data['info']['energy_usage']
        
        all_max_x_positions.append(max_x_pos)
        cave_max_x_positions[cave_id].append(max_x_pos)
        all_energy_usage.append(last_energy_usage)
        cave_energy_usage[cave_id].append(last_energy_usage)

print(f"Processed {len(all_max_x_positions)} episodes across {len(cave_max_x_positions)} caves")

MAXIMUM X-POSITION AND ENERGY USAGE ANALYSIS
Processing Cave 271...
Processing Cave 272...
Processing Cave 273...
Processing Cave 274...
Processing Cave 275...
Processing Cave 276...
Processing Cave 277...
Processing Cave 278...
Processing Cave 279...
Processing Cave 280...
Processing Cave 281...
Processing Cave 282...
Processing Cave 283...
Processing Cave 284...
Processing Cave 285...
Processing Cave 286...
Processing Cave 287...
Processing Cave 288...
Processing Cave 289...
Processing Cave 290...
Processing Cave 291...
Processing Cave 292...
Processing Cave 293...
Processing Cave 294...
Processing Cave 295...
Processing Cave 296...
Processing Cave 297...
Processing Cave 298...
Processing Cave 299...
Processing Cave 300...
Processed 90 episodes across 30 caves


In [7]:
# Calculate per-cave statistics for max x-position
cave_max_x_stats = {}
cave_max_x_means = []

print("PER-CAVE MAX X-POSITION STATISTICS")
print("-" * 80)
print(f"{'Cave ID':<8} {'Mean Max X':<12} {'Std Dev':<12} {'Episodes':<10} {'Min':<12} {'Max':<12}")
print("-" * 80)

for cave_id in sorted(cave_max_x_positions.keys()):
    max_x_values = cave_max_x_positions[cave_id]
    cave_mean_max_x = np.mean(max_x_values)
    cave_std_max_x = np.std(max_x_values)
    cave_min_max_x = np.min(max_x_values)
    cave_max_max_x = np.max(max_x_values)
    
    cave_max_x_stats[cave_id] = {
        'mean': cave_mean_max_x,
        'std': cave_std_max_x,
        'min': cave_min_max_x,
        'max': cave_max_max_x,
        'count': len(max_x_values)
    }
    
    cave_max_x_means.append(cave_mean_max_x)
    
    print(f"{cave_id:<8} {cave_mean_max_x:<12.6f} {cave_std_max_x:<12.6f} {len(max_x_values):<10} {cave_min_max_x:<12.6f} {cave_max_max_x:<12.6f}")

print("-" * 80)

# Calculate per-cave statistics for energy usage
cave_energy_stats = {}
cave_energy_means = []

print("PER-CAVE ENERGY USAGE STATISTICS")
print("-" * 80)
print(f"{'Cave ID':<8} {'Mean Energy':<12} {'Std Dev':<12} {'Episodes':<10} {'Min':<12} {'Max':<12}")
print("-" * 80)

for cave_id in sorted(cave_energy_usage.keys()):
    energy_values = cave_energy_usage[cave_id]
    cave_mean_energy = np.mean(energy_values)
    cave_std_energy = np.std(energy_values)
    cave_min_energy = np.min(energy_values)
    cave_max_energy = np.max(energy_values)
    
    cave_energy_stats[cave_id] = {
        'mean': cave_mean_energy,
        'std': cave_std_energy,
        'min': cave_min_energy,
        'max': cave_max_energy,
        'count': len(energy_values)
    }
    
    cave_energy_means.append(cave_mean_energy)
    
    print(f"{cave_id:<8} {cave_mean_energy:<12.6f} {cave_std_energy:<12.6f} {len(energy_values):<10} {cave_min_energy:<12.6f} {cave_max_energy:<12.6f}")

print("-" * 80)

PER-CAVE MAX X-POSITION STATISTICS
--------------------------------------------------------------------------------
Cave ID  Mean Max X   Std Dev      Episodes   Min          Max         
--------------------------------------------------------------------------------
271      7.748396     1.407501     3          5.767320     8.906534    
272      10.237386    1.623497     3          8.239931     12.216545   
273      6.632163     4.278160     3          0.582394     9.721834    
274      8.685394     0.251566     3          8.469632     9.038251    
275      9.657146     0.977523     3          8.513836     10.901823   
276      5.454879     3.300571     3          1.956897     9.880387    
277      6.752519     3.396132     3          1.952688     9.300028    
278      8.417278     1.811422     3          6.932600     10.967564   
279      10.149696    5.398826     3          5.260494     17.672955   
280      9.850865     2.503922     3          6.596302     12.686575   
281      6.

In [8]:
# Calculate comprehensive metrics per cave and save to JSON
print("="*60)
print("COMPREHENSIVE CAVE METRICS ANALYSIS")
print("="*60)

# Use data already calculated in previous cells
comprehensive_cave_metrics = {}

for cave_id in sorted(cave_rewards.keys()):
    comprehensive_cave_metrics[str(cave_id)] = {
        'rewards': {
            'average': float(cave_stats[cave_id]['mean']),
            'std_deviation': float(cave_stats[cave_id]['std']),
            'count': cave_stats[cave_id]['count'],
            'min': float(cave_stats[cave_id]['min']),
            'max': float(cave_stats[cave_id]['max'])
        },
        'energy_consumptions': {
            'average': float(cave_energy_stats[cave_id]['mean']),
            'std_deviation': float(cave_energy_stats[cave_id]['std']),
            'count': cave_energy_stats[cave_id]['count'],
            'min': float(cave_energy_stats[cave_id]['min']),
            'max': float(cave_energy_stats[cave_id]['max'])
        },
        'max_x_positions': {
            'average': float(cave_max_x_stats[cave_id]['mean']),
            'std_deviation': float(cave_max_x_stats[cave_id]['std']),
            'count': cave_max_x_stats[cave_id]['count'],
            'min': float(cave_max_x_stats[cave_id]['min']),
            'max': float(cave_max_x_stats[cave_id]['max'])
        },
        'steps': {
            'average': float(np.mean([ep['steps'] for cave_data in session_data['caves'] 
                                    if cave_data['cave_id'] == cave_id 
                                    for ep in cave_data['episodes']])),
            'std_deviation': float(np.std([ep['steps'] for cave_data in session_data['caves'] 
                                          if cave_data['cave_id'] == cave_id 
                                          for ep in cave_data['episodes']])),
            'count': len([ep['steps'] for cave_data in session_data['caves'] 
                         if cave_data['cave_id'] == cave_id 
                         for ep in cave_data['episodes']]),
            'min': float(np.min([ep['steps'] for cave_data in session_data['caves'] 
                                if cave_data['cave_id'] == cave_id 
                                for ep in cave_data['episodes']])),
            'max': float(np.max([ep['steps'] for cave_data in session_data['caves'] 
                                if cave_data['cave_id'] == cave_id 
                                for ep in cave_data['episodes']]))
        }
    }

# Display comprehensive results
print("\nPer-Cave Comprehensive Metrics Summary:")
print("=" * 70)
for cave_id in sorted(comprehensive_cave_metrics.keys(), key=int):
    metrics = comprehensive_cave_metrics[cave_id]
    print(f"\nCave {cave_id}:")
    print(f"  Episodes: {metrics['rewards']['count']}")
    print(f"  Reward:             {metrics['rewards']['average']:.3f} ± {metrics['rewards']['std_deviation']:.3f}")
    print(f"  Energy Consumption: {metrics['energy_consumptions']['average']:.3f} ± {metrics['energy_consumptions']['std_deviation']:.3f}")
    print(f"  Max X-Position:     {metrics['max_x_positions']['average']:.6f} ± {metrics['max_x_positions']['std_deviation']:.6f}")
    print(f"  Steps:              {metrics['steps']['average']:.1f} ± {metrics['steps']['std_deviation']:.1f}")

# Save comprehensive metrics to JSON file in the evaluation session folder
session_folder = Path(eval_session_path)
output_file = session_folder / "cave_metrics_comprehensive.json"

# Add metadata to the JSON
output_data = {
    'metadata': {
        'evaluation_session': eval_session_path.split('/')[-1],
        'timestamp': session_data.get('eval_timestamp', 'unknown'),
        'caves_directory': session_data.get('caves_directory', 'unknown'),
        'episodes_per_cave': session_data.get('episodes_per_cave', 0),
        'max_steps': session_data.get('max_steps', 0),
        'total_caves': len(comprehensive_cave_metrics),
        'total_episodes': sum(metrics['rewards']['count'] for metrics in comprehensive_cave_metrics.values())
    },
    'cave_metrics': comprehensive_cave_metrics
}

with open(output_file, 'w') as f:
    json.dump(output_data, f, indent=2)

print(f"\nComprehensive metrics saved to: {output_file}")
print(f"Total caves analyzed: {len(comprehensive_cave_metrics)}")

# Calculate overall averages across all caves
print("\n" + "="*60)
print("OVERALL AVERAGES ACROSS ALL CAVES")
print("="*60)
all_cave_reward_avgs = [metrics['rewards']['average'] for metrics in comprehensive_cave_metrics.values()]
all_cave_energy_avgs = [metrics['energy_consumptions']['average'] for metrics in comprehensive_cave_metrics.values()]
all_cave_max_x_avgs = [metrics['max_x_positions']['average'] for metrics in comprehensive_cave_metrics.values()]
all_cave_steps_avgs = [metrics['steps']['average'] for metrics in comprehensive_cave_metrics.values()]

print(f"Average Reward across caves:       {np.mean(all_cave_reward_avgs):.3f} ± {np.std(all_cave_reward_avgs):.3f}")
print(f"Average Energy across caves:       {np.mean(all_cave_energy_avgs):.3f} ± {np.std(all_cave_energy_avgs):.3f}")
print(f"Average Max X-Position across caves: {np.mean(all_cave_max_x_avgs):.6f} ± {np.std(all_cave_max_x_avgs):.6f}")
print(f"Average Steps across caves:        {np.mean(all_cave_steps_avgs):.1f} ± {np.std(all_cave_steps_avgs):.1f}")
print("="*60)

COMPREHENSIVE CAVE METRICS ANALYSIS

Per-Cave Comprehensive Metrics Summary:

Cave 271:
  Episodes: 3
  Reward:             35.243 ± 4.857
  Energy Consumption: 101186.635 ± 14870.215
  Max X-Position:     7.748396 ± 1.407501
  Steps:              2764.0 ± 885.5

Cave 272:
  Episodes: 3
  Reward:             30.657 ± 11.144
  Energy Consumption: 168114.591 ± 40884.405
  Max X-Position:     10.237386 ± 1.623497
  Steps:              6062.7 ± 1953.3

Cave 273:
  Episodes: 3
  Reward:             17.902 ± 12.773
  Energy Consumption: 143698.470 ± 95668.777
  Max X-Position:     6.632163 ± 4.278160
  Steps:              5052.3 ± 3461.5

Cave 274:
  Episodes: 3
  Reward:             31.085 ± 10.200
  Energy Consumption: 115290.297 ± 39378.304
  Max X-Position:     8.685394 ± 0.251566
  Steps:              3761.0 ± 587.4

Cave 275:
  Episodes: 3
  Reward:             34.691 ± 14.786
  Energy Consumption: 194892.620 ± 46687.003
  Max X-Position:     9.657146 ± 0.977523
  Steps:              6

In [9]:
# Create LaTeX-ready table from comprehensive cave metrics
print("="*60)
print("GENERATING LATEX TABLE")
print("="*60)

# Generate LaTeX table
latex_table = []
latex_table.append("\\begin{table}[htbp]")
latex_table.append("\\centering")
latex_table.append("\\caption{Comprehensive Cave Performance Metrics}")
latex_table.append("\\label{tab:cave_metrics}")
latex_table.append("\\begin{tabular}{c|cccc}")
latex_table.append("\\toprule")
latex_table.append("\\textbf{Cave ID} & \\textbf{Reward} & \\textbf{Energy} & \\textbf{Max X-Position} & \\textbf{Steps} \\\\")
latex_table.append("\\midrule")

# Add data rows
for cave_id in sorted(comprehensive_cave_metrics.keys(), key=int):
    metrics = comprehensive_cave_metrics[cave_id]
    
    reward_str = f"{metrics['rewards']['average']:.2f} ± {metrics['rewards']['std_deviation']:.2f}"
    energy_str = f"{metrics['energy_consumptions']['average']:.2f} ± {metrics['energy_consumptions']['std_deviation']:.2f}"
    max_x_str = f"{metrics['max_x_positions']['average']:.4f} ± {metrics['max_x_positions']['std_deviation']:.4f}"
    steps_str = f"{metrics['steps']['average']:.0f} ± {metrics['steps']['std_deviation']:.0f}"
    
    row = f"{cave_id} & {reward_str} & {energy_str} & {max_x_str} & {steps_str} \\\\"
    latex_table.append(row)

latex_table.append("\\bottomrule")
latex_table.append("\\end{tabular}")
latex_table.append("\\end{table}")

# Print the LaTeX table
print("LaTeX Table Code:")
print("=" * 40)
for line in latex_table:
    print(line)

# Save LaTeX table to file
latex_file = session_folder / "cave_metrics_latex_table.tex"
with open(latex_file, 'w') as f:
    f.write('\n'.join(latex_table))

print("\n" + "="*40)
print(f"LaTeX table saved to: {latex_file}")



GENERATING LATEX TABLE
LaTeX Table Code:
\begin{table}[htbp]
\centering
\caption{Comprehensive Cave Performance Metrics}
\label{tab:cave_metrics}
\begin{tabular}{c|cccc}
\toprule
\textbf{Cave ID} & \textbf{Reward} & \textbf{Energy} & \textbf{Max X-Position} & \textbf{Steps} \\
\midrule
271 & 35.24 ± 4.86 & 101186.64 ± 14870.21 & 7.7484 ± 1.4075 & 2764 ± 885 \\
272 & 30.66 ± 11.14 & 168114.59 ± 40884.40 & 10.2374 ± 1.6235 & 6063 ± 1953 \\
273 & 17.90 ± 12.77 & 143698.47 ± 95668.78 & 6.6322 ± 4.2782 & 5052 ± 3462 \\
274 & 31.09 ± 10.20 & 115290.30 ± 39378.30 & 8.6854 ± 0.2516 & 3761 ± 587 \\
275 & 34.69 ± 14.79 & 194892.62 ± 46687.00 & 9.6571 ± 0.9775 & 6171 ± 1879 \\
276 & 17.42 ± 11.67 & 87735.22 ± 61206.99 & 5.4549 ± 3.3006 & 1563 ± 1374 \\
277 & 28.28 ± 19.53 & 118638.60 ± 61921.47 & 6.7525 ± 3.3961 & 3043 ± 2017 \\
278 & 13.67 ± 19.32 & 160454.47 ± 51642.95 & 8.4173 ± 1.8114 & 5549 ± 2069 \\
279 & 40.02 ± 20.91 & 136396.48 ± 68901.99 & 10.1497 ± 5.3988 & 3712 ± 2655 \\
280 & 31.62 ±